(Introduction_Attributes)=
# Attributes

MolSysMT attributes describe non-exclusive layers of a molecular system. Stable topological attributes identify atoms and semantic hierarchy. Structural attributes describe geometry and trajectories. `atom_index` and `n_atoms` belong to both layers: they define the stable atom inventory and the atom axis shared by structural arrays. Mechanical attributes contain force-field data. Chemical-state attributes describe chemistry that may differ between states, including bonds, formal charge, aromaticity, radicals, implicit hydrogens, stereochemistry, and covalent components.

The topological `atom_type` is the chemical element symbol, such as `C`, `Ca` or `Cl`. It is distinct from the molecular atom name (`atom_name`) and the mechanical force-field assignment (`atom_ff_type`). Isotope mass numbers are stored separately as nullable `isotope` values; they are dimensionless integers, not atomic masses. The explicit aliases `D` and `T` normalize to `H` with mass numbers 2 and 3. Capitalization carries meaning: `CA` is not the symbol for calcium. See {ref}`Tutorial_Atom_Types` for validation and normalization.


A molecular form does not need to contain every attribute. Use {func}`molsysmt.basic.has_attribute` to distinguish information stored in a particular object from information supported by its form. By default it reports instance presence, so an optional temperature, occupancy, component label, or chemical-state ID that is absent returns `False`; `include_none=True` asks only whether the form supports the attribute. For `MolSys`, this check composes its topology, structures, and molecular-mechanics form contracts rather than adding methods to the native component classes. Native objects resolve chemical-state attributes through `chemical_state='reference'`, `chemical_state='structure'`, or an explicit 0-based state index. The explicit index is execution-local: it does not mutate the stored reference. The `'structure'` resolver uses the nullable `structure_chemical_state_index` association owned by `MolSys` and requires the selected structures to share one known state. State IDs are labels and are not selectors because duplicates are valid.

An empty native bond collection is valid data: per-atom bond counts are zero and per-atom bond-index collections are empty. A missing attribute can instead return `None`. Neither case establishes complete chemical connectivity. Candidate connectivity is separate inferred evidence: geometric uniqueness must be checked against the source population, not only the selected output. Group indices describe storage order; they do not necessarily describe polymer sequence. An optional distance-based backbone candidate policy preserves the source axes and reports competing partners and incomplete geometry rather than relabeling or applying bonds. See {ref}`Tutorial_Peptide_Bond_Candidates`. Specify `element='atom'` for per-atom values; the default system-level query returns aggregate values. See {ref}`Getting attributes <Tutorial_Get>`.

Observed H-parent template candidates are a separate layer of inferred evidence. A local consensus on the heavy parent of each exact H name need not establish one compatible whole-group H inventory, and neither establishes protonation or correct H placement. The candidate report preserves those distinctions without editing stored bonds or coordinates. See {ref}`Tutorial_Covalent_Bond_Candidates`.

Group-level bond counts describe distinct bonds incident on each group.
Internal bonds count once and bonds between groups count once for each
participating group. These counts belong to the selected chemical state,
so the same stable group inventory may have different counts across states.

Native covalent inference applies bounded candidate evidence to a detached system and archives its original atom/bond/structure/state domains in ChemicalStates preparation history. Current bond rows retain declared or inferred evidence and a history reference; extraction remaps current rows but does not rewrite historical indices. Added edges leave completeness partial and invalidate output interaction occurrences. The PDB reader can explicitly request MolSysMT or OpenMM; default optional-engine failures are diagnosed and archived. See {ref}`Tutorial_Infer_Covalent_Bonds`.


Canonical attribute names are singular, such as `atom_name`, but selected plural and anatomical aliases such as `atom_names` and `residue_name` are accepted for convenience. With an explicit element, short names are contextual: `msm.get(system, element='group', name=True)` requests `group_name`. An alias and its canonical name are alternatives and must not be supplied together. Integrations can inspect the versioned mapping with `msm.attribute.get_argument_aliases()` without changing MolSysMT's runtime configuration.

Alternate sites are sparse structural evidence: each structure has a dictionary keyed by integer atom indices in that source system. The entry's `atom_id` values are string labels for the deposited sites. Selecting atoms filters the source-index keys without changing their positions; selecting structures changes the outer list in the requested order. An empty dictionary denotes a structure with no alternate sites, while `None` denotes absent evidence.

Persisted alternate geometry must carry its physical units independently of the application's current unit policy. Structures YAML 0.2 uses verified quantity records for the main structural arrays and for alternate-site coordinates and B factors. Its readers check field identity, physical dimensions and record integrity. Atom selections creating a new structural object remap site keys to that object's local atom axis; IDs stay labels. See {ref}`Structures YAML <Tutorial_Form_file_structures_yaml>` for supported versions and migration.


## Attributes you can get

The following table shows the list of attribute arguments you can use in {func}`molsysmt.basic.get()`, together with their meaning and the list of elements they can be used with.

In [1]:
import molsysmt as msm
import pandas as pd
from IPython.display import HTML

rows = []
for name, meta in sorted(msm.attribute.attributes.items()):
    elements = meta.get('get_from', [])
    elem_str = ', '.join([f'<code>{e}</code>' for e in elements]) if elements else ''
    
    cats = []
    if meta.get('topological'): cats.append('Topological')
    if meta.get('structural'): cats.append('Structural')
    if meta.get('mechanical'): cats.append('Mechanical')
    if meta.get('chemical_state'): cats.append('Chemical State')
    cat_str = ', '.join(cats) if cats else 'General'
    
    rows.append({
        'Property': f'<code>{name}</code>',
        'Category': cat_str,
        'Elements': elem_str
    })

df = pd.DataFrame(rows)

html = '<table class="table"><thead><tr>'
html += ''.join([f'<th style="text-align: left;">{col}</th>' for col in df.columns])
html += '</tr></thead><tbody>'
for _, row in df.iterrows():
    html += '<tr>'
    for col in df.columns:
        html += f'<td style="text-align: left;">{row[col]}</td>'
    html += '</tr>'
html += '</tbody></table>'
HTML(html)

Property,Category,Elements
allows_implicit_hydrogens,Chemical State,atom
alternate_location,Structural,"atom, system"
atom_ff_type,Mechanical,"atom, system"
atom_id,Topological,"atom, group, component, molecule, chain, entity"
atom_index,"Topological, Structural","atom, group, component, molecule, chain, entity"
atom_is_aromatic,Chemical State,atom
atom_name,Topological,"atom, group, component, molecule, chain, entity"
atom_stereochemistry,Chemical State,atom
atom_type,Topological,"atom, group, component, molecule, chain, entity"
b_factor,Structural,"atom, system"
